<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/LocalLLM_Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 라이브러리를 설치한다.*이탤릭체 텍스트*

In [1]:
pip install -q requests openpyxl

필요한 모듈을 임포트 한다.

In [ ]:
from pathlib import Path
from openpyxl import Workbook, load_workbook
import requests
import json

C:\VirtualC에 가상의 컴퓨터 폴더공간과 파일을 만든다.

In [ ]:
ROOT = Path(r"C:\VirtualC")
# --------------------------------------------------
# 1. 실습용 가상 C 드라이브 만들기
# --------------------------------------------------

(ROOT / "Reports").mkdir(parents=True, exist_ok=True)
(ROOT / "Finance").mkdir(parents=True, exist_ok=True)
(ROOT / "AI").mkdir(parents=True, exist_ok=True)

(ROOT / "Reports" / "AI_report.txt").write_text(
    "이 파일은 AI 도입 보고서입니다.",
    encoding="utf-8"
)

(ROOT / "Reports" / "meeting_memo.txt").write_text(
    "회의 메모입니다. 예산과 일정 논의가 있었습니다.",
    encoding="utf-8"
)

(ROOT / "AI" / "ollama_note.txt").write_text(
    "Ollama와 Gemma 모델을 이용한 로컬 LLM 실습 메모입니다.",
    encoding="utf-8"
)

wb = Workbook()
ws = wb.active
ws.title = "Sales"
ws.append(["항목", "내용", "금액"])
ws.append(["매출", "2026년 1분기 매출 데이터", 1000])
ws.append(["비용", "마케팅 비용", 300])
wb.save(ROOT / "Finance" / "sales_2026.xlsx")

wb = Workbook()
ws = wb.active
ws.title = "LLM"
ws.append(["모델", "설명"])
ws.append(["Gemma", "가벼운 로컬 LLM 실습용 모델"])
ws.append(["Qwen", "성능은 좋지만 무거운 모델"])
wb.save(ROOT / "AI" / "local_llm_models.xlsx")

로컬에서 키워드로 파일 찾는 함수 구성

In [ ]:
def search_local_files(mode, keyword):
    results = []

    if mode == "filename":
        for file_path in ROOT.rglob("*"):
            if file_path.is_file():
                if keyword.lower() in file_path.name.lower():
                    results.append(
                        f"{file_path.relative_to(ROOT)}"
                    )

    elif mode == "excel_content":
        for file_path in ROOT.rglob("*.xlsx"):
            wb = load_workbook(
                file_path,
                read_only=True,
                data_only=True
            )

            for sheet in wb.worksheets:
                for row in sheet.iter_rows():
                    for cell in row:
                        if cell.value is None:
                            continue

                        cell_text = str(cell.value)

                        if keyword.lower() in cell_text.lower():
                            results.append(
                                f"{file_path.relative_to(ROOT)} / {sheet.title}!{cell.coordinate} = {cell_text}"
                            )

            wb.close()

    return results

ollama 모델과 기본 URL을 설정한다.

In [ ]:
MODEL = "gemma3:4b"
OLLAMA_URL = "http://localhost:11434/api/chat"

ollama를 이용해 명령어를 구성하는 함수

In [ ]:
def ask_ollama(messages, json_mode=False):
    body = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

    if json_mode:
        body["format"] = "json"

    response = requests.post(
        OLLAMA_URL,
        json=body
    )

    return response.json()["message"]["content"]

명령 Prompt를 구성한다.

In [ ]:
planner_prompt = """
너는 사용자의 한국어 파일 검색 요청을 JSON으로 바꾸는 분류기다.

반드시 JSON만 출력하라.

가능한 mode는 두 가지다.

1. filename
- 파일 이름에 keyword가 들어간 파일을 찾는다.
- 예: "AI 들어가는 파일 찾아줘"
- 예: "보고서라는 이름의 파일 찾아줘"

2. excel_content
- 엑셀 파일 내부 셀 값에 keyword가 쓰인 파일을 찾는다.
- 예: "매출 단어가 쓰인 엑셀 파일 찾아줘"
- 예: "Gemma가 들어간 액셀 파일 찾아줘"

출력 형식:
{
  "mode": "filename",
  "keyword": "AI"
}

규칙:
- keyword에는 조사와 불필요한 말을 제거한다.
- "엑셀", "액셀", "xlsx", "단어가 쓰인", "내용에 들어간"이 있으면 excel_content로 본다.
- 단순히 "들어가는 파일", "이름에 들어간 파일"이면 filename으로 본다.
"""

답변  Prompt를 구성한다.

In [ ]:
answer_prompt = """
너는 로컬 파일 검색 결과를 사용자에게 설명하는 AI다.
검색 결과에 없는 파일은 지어내지 마라.
짧고 명확하게 한국어로 답하라.
"""

이제 무한 루프돌며 명령에 따라 검색을 실시. exit를 입력하면 종료

In [ ]:
print("로컬 파일 검색 RAG 실습 시작- 종료시 exit 입력")
print()
print("예시 요청:")
print("- AI 들어가는 파일 찾아줘")
print("- 매출 단어가 쓰인 엑셀 파일 찾아줘")
print("- Gemma 단어가 쓰인 액셀 파일 찾아줘")
print("- exit")

while True:
    user_query = input("\n요청: ").strip()

    if user_query.lower() == "exit":
        print("종료합니다.")
        break

    # 1단계: LLM이 사용자 요청을 검색 계획으로 변환
    planner_messages = [
        {
            "role": "system",
            "content": planner_prompt
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    plan_text = ask_ollama(
        planner_messages,
        json_mode=True
    )

    plan = json.loads(plan_text)

    mode = plan["mode"]
    keyword = plan["keyword"]

    print("\n[LLM 검색 의도 해석]")
    print(plan)

    # 2단계: Python 함수가 실제 로컬 파일 검색 수행
    results = search_local_files(
        mode,
        keyword
    )

    if len(results) == 0:
        search_context = "검색 결과 없음"
    else:
        search_context = "\n".join(results)

    print("\n[Python 검색 결과]")
    print(search_context)

    # 3단계: 검색 결과를 LLM에 다시 넣고 최종 답변 생성
    answer_messages = [
        {
            "role": "system",
            "content": answer_prompt
        },
        {
            "role": "user",
            "content": f"""
사용자 요청:
{user_query}

검색 위치:
{ROOT}

LLM이 해석한 검색 계획:
{plan}

Python이 실제 검색한 결과:
{search_context}

위 검색 결과만 근거로 최종 답변을 작성하라.
"""
        }
    ]

    answer = ask_ollama(answer_messages)

    print("\n[LLM 최종 답변]")
    print(answer)